# 📗 지식그래프 적재: 파일에서 그래프로

앞 교안에서 **모양**을 정했습니다. 무엇을 노드로 둘지, 날짜를 어디에 둘지, 관계 타입을 어떻게 나눌지, 무엇을 키로 삼을지. 이제 그 모양대로 **실제로 넣습니다.** 파일에 담긴 **노드 15,540개와 관계 91,966개**가 오늘 그래프로 들어갑니다.

넣는 길은 **두 가지**입니다.

| 길 | 파일을 읽는 주체 | 이 교안에서 |
|---|---|---|
| `LOAD CSV`·`apoc.load.json` | **서버**가 `import` 폴더의 파일을 읽는다 | 1~4절 |
| 파이썬 + `UNWIND` | **파이썬**이 읽어 목록으로 보낸다 | 5절 |

두 길은 할 수 있는 일이 다릅니다. 마지막 절에서 **언제 어느 쪽을 쓰는지** 기준을 세웁니다.

> **데이터 출처**: 아래 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다. 교육용으로 지어낸 값이 없습니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | CC0 |
> | 서울교통공사 역간거리·소요시간 (`seoul_metro_stations_cp949.csv`) | 서울 열린데이터광장 OA-12034 | 공공누리 1유형(출처표시) |
> | 서울교통공사 환승역 환승인원 (`seoul_metro_transfer_cp949.csv`) | 서울 열린데이터광장 OA-12033 | 공공누리 1유형(출처표시) |
>
> 의료 그래프는 **2016년에 정리된 자료**입니다. 그래서 "이 약이 이 병에 쓰인다고 **문헌에 정리돼 있다**"까지가 이 데이터가 말하는 것이고, "효능이 입증됐다"는 아닙니다. 지식그래프를 다룰 때 이 구분을 놓치면 안 됩니다.
>
> 지하철 CSV 두 개는 포털에서 받은 **바이트 그대로**라 인코딩이 `CP949` 입니다. UTF-8 로 읽으면 글자가 깨집니다.

## ⏪ 복습: 지난 시간까지

- **적재 순서**(교안_01): 제약을 먼저, 노드를 다 넣고, 관계를 마지막에.
- **노드냐 속성이냐**(교안_02 3절): 값을 **거쳐 건너갈** 일이 있으면 노드. 재위를 왕의 속성이 아니라 노드로 뺐습니다.
- **관계 타입**(교안_02 5절): 뜻이 다르면 나눕니다. 생부와 양자를 한 타입에 담지 않았듯이. 그 결정이 오늘 **적재 속도**로 돌아옵니다.
- **키**(교안_02 8절): 이름은 유일 키가 아닙니다. 묘호와 휘가 따로였듯, 오늘 쓰는 의료 데이터도 이름이 아니라 `id` 로 `MERGE` 합니다.

지난 교안은 **작은 실험 노드**만 만들고 지웠습니다. 오늘 넣는 것이 이 단원의 진짜 그래프입니다.

**오늘의 목표**

- [ ] 파일을 먼저 열어 머리글·인코딩·행 수를 확인하고, `CP949` 를 UTF-8 로 바꿔 둔다.
- [ ] 적재 **전에** 레이블마다 제약조건을 걸고, `LOAD CSV` + `IN TRANSACTIONS` 로 멱등 적재한다.
- [ ] **인덱스가 레이블별로 걸린다**는 것을 적재 시간으로 직접 잰다.
- [ ] **APOC** 로 JSON 명세서를 읽어 적재 결과를 대조한다.
- [ ] 같은 파일을 **파이썬이 읽어 `UNWIND` 로** 보내 보고, 두 길이 같은 곳에 도착하는지 확인한다.
- [ ] `LOAD CSV` 와 **파이썬이 읽어 보내는 길** 중 언제 어느 쪽을 쓰는지 기준을 세운다.

아래 준비 셀들을 위에서부터 실행하세요. **연결 셀은 이해하지 않아도 됩니다.** Neo4j 는 반드시 **실습 전용 DB**에 연결하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

### 오늘 쓰는 APOC 는 무엇인가요?

**APOC**(Awesome Procedures On Cypher)는 Neo4j 의 **확장 라이브러리**입니다. Cypher 만으로는 못 하거나 길게 써야 하는 일을 **한 줄로** 줄여 줍니다. Neo4j 에 기본으로 들어 있지 않아서 인스턴스마다 **플러그인으로 설치**합니다(Desktop 은 인스턴스의 **Plugins**에서).

오늘 쓰는 것은 셋뿐입니다.

| 부르는 이름 | 하는 일 | 쓰는 곳 |
|---|---|---|
| `apoc.version()` | 설치됐는지, 어느 버전인지 알려 준다 | 바로 아래 환경 점검 |
| `apoc.load.json('file:///파일.json')` | JSON 파일을 읽어 한 건씩 돌려준다 | 4절 · 과제 LV1·LV2 |
| `apoc.meta.stats()` | 지금 그래프에 무엇이 얼마나 있는지 요약한다 | 4절 · 과제 LV1 |

> **함수**는 값 하나를 돌려주어 `RETURN`·`WHERE` 처럼 식이 오는 자리에 그대로 씁니다. **프로시저**는 값이 아니라 **행을 여러 개** 흘려보내서, `CALL ... YIELD` 로 받아 쿼리 흐름에 이어 붙입니다. `RETURN apoc.version()` 과 `CALL apoc.load.json(...) YIELD value` 의 차이가 그것입니다.

### 서버의 폴더 경로 확인하기

`import` 폴더가 어디인지는 **서버에 물어보면** 됩니다. `SHOW SETTINGS` 가 서버 설정값을 돌려줍니다.

In [ ]:
# [제공 코드] 이 서버가 쓰는 폴더 경로를 물어본다(아래 apoc.conf 셀이 이 값을 쓴다)
dirs = {row['name']: row['value'] for row in run_cypher(
    "SHOW SETTINGS YIELD name, value "
    "WHERE name IN ['server.directories.neo4j_home', 'server.directories.import'] "
    "RETURN name, value")}
for name, value in sorted(dirs.items()):
    print(name, '=', value)

In [ ]:
# [제공 코드] 데이터 파일을 Neo4j import 폴더로 복사: 이 셀은 실행만 하세요.
# LOAD CSV·apoc.load.json 은 보안상 서버의 import 폴더 안 파일만 읽습니다.
# - .env 에 NEO4J_IMPORT_DIR 이 있으면 data/ 의 파일을 자동 복사합니다.
# - 없으면(수동 복사한 경우) 그대로 넘어갑니다. Neo4j Desktop 은 인스턴스 메뉴의
#   "Open folder > Import" 로 폴더를 열어 data/ 의 파일을 직접 복사해 두세요.
import shutil
from pathlib import Path

_IMPORT_DIR = os.getenv("NEO4J_IMPORT_DIR", "")
_SRC_DIR = Path("data") if Path("data").exists() else Path("../data")
if _IMPORT_DIR:
    # csv 와 json 만 복사합니다. 다음 준비 셀이 이 파일들을 file:/// 로 읽습니다
    for f in sorted(_SRC_DIR.glob("*.csv")) + sorted(_SRC_DIR.glob("*.json")):
        shutil.copy(f, Path(_IMPORT_DIR) / f.name)
        print("복사:", f.name)
else:
    print("NEO4J_IMPORT_DIR 미설정: data/ 의 파일을 import 폴더에 직접 복사했는지 확인하세요.")

> `LOAD CSV` 와 `apoc.load.json` 은 보안상 **서버의 `import` 폴더** 안 파일만 읽습니다. 위 복사 셀이 `.env` 의 `NEO4J_IMPORT_DIR` 로 자동 복사해 줍니다(자기 로컬에서는 Neo4j 의 `import` 폴더에 직접 복사). 그래서 경로를 **`file:///파일명.csv`** 로 씁니다.

> **`apoc.load.json` 은 따로 켜 줘야 합니다.** APOC 를 설치해도 파일 읽기는 기본으로 꺼져 있습니다(`LOAD CSV` 를 여는 설정과는 별개입니다). 서버의 `conf` 폴더에 **`apoc.conf`** 를 만들어 두 줄을 적어야 하는데, 아래 셀이 위에서 찾은 경로를 그대로 써서 그 파일을 만듭니다. **Neo4j 가 이 컴퓨터에 있을 때만** 됩니다(원격 서버라면 그 서버에서 직접 만드세요). 파일이 이미 있으면 이 두 줄로 덮어씁니다.

In [ ]:
# [제공 코드] apoc.conf 를 만들어 apoc.load.json 의 파일 읽기를 켠다
# 서버가 시작할 때 읽는 파일이라, 만든 뒤에는 Neo4j 인스턴스를 껐다 켜야 적용된다
from pathlib import Path

import_dir = Path(dirs['server.directories.import'])
apoc_conf = import_dir.parent / 'conf' / 'apoc.conf'   # import 와 나란히 있는 conf 폴더
apoc_conf.write_text('apoc.import.file.enabled=true\n'
                     'apoc.import.file.use_neo4j_config=true\n', encoding='utf-8')
print('만들었습니다:', apoc_conf)

In [ ]:
# [제공 코드] 환경 점검: 이 셀은 실행만 하세요.
# APOC · import 폴더 · APOC 파일 권한 셋을 미리 확인한다
try:
    # apoc.version() 은 APOC 가 깔려 있을 때만 있는 함수다
    apoc_version = run_cypher("RETURN apoc.version() AS v")[0]['v']
    print(f"APOC 설치됨: {apoc_version}")
except Exception as error:
    print("APOC 를 찾지 못했습니다. Neo4j Desktop 인스턴스의 Plugins 에서 APOC 를 설치하고 재시작하세요.")
    print("  원문:", error)

In [ ]:
# [제공 코드] (이어서)
# 노드는 만들지 않고 행 수만 센다. 파일을 읽을 수 있는지만 보는 것이다
try:
    probe = run_cypher("LOAD CSV WITH HEADERS FROM 'file:///hetionet_nodes.csv' AS row "
                       "RETURN count(row) AS n")[0]['n']
    print(f"import 폴더에서 파일을 읽었습니다: {probe}행")
except Exception as error:
    print("LOAD CSV 가 파일을 읽지 못했습니다. 파일이 import 폴더에 복사됐는지(위 복사 셀) 확인하세요.")
    print("  원문:", error)

In [ ]:
# [제공 코드] (이어서)
# apoc.load.json 은 LOAD CSV 와 권한이 달라서(apoc.conf 소관) 따로 확인한다
try:
    n_spec = run_cypher("CALL apoc.load.json('file:///hetionet_relspec.json') YIELD value "
                        "RETURN count(value) AS n")[0]['n']
    print(f"APOC 로 JSON 을 읽었습니다: {n_spec}건")
except Exception as error:
    print("APOC 가 파일을 읽지 못했습니다. 위 apoc.conf 셀을 실행하고 인스턴스를 껐다 켜세요.")
    print("  원문:", error)

---
# 1. 파일부터 열어 봅니다

## 왜 필요할까요?
적재 코드를 쓰기 전에 **파일이 어떻게 생겼는지** 봐야 합니다. 확인할 것은 세 가지입니다.

1. **머리글(칸 이름)**: 이 이름이 곧 `row['이름']` 의 키가 됩니다.
2. **인코딩**: 국내 공공데이터는 `UTF-8` 과 `CP949` 가 섞여 있습니다. 잘못 읽으면 글자가 깨집니다.
3. **행 수**: 적재가 끝난 뒤 "몇 개가 들어갔나"를 대조할 기준입니다.

## 오늘의 파일

| 파일 | 칸 | 행 수 | 인코딩 |
|---|---|---|---|
| `hetionet_nodes.csv` | `id`, `name`, `label` | 15,540 | UTF-8 |
| `hetionet_edges.csv` | `source`, `rel`, `target` | 91,966 | UTF-8 |
| `seoul_metro_stations_cp949.csv` | `연번`, `호선`, `역명`, `소요시간`, `역간거리(km)`, `호선별누계(km)` | 279 | **CP949** |

노드 파일의 `id` 는 `Compound::DB00014` 처럼 **어디서 온 무엇인지**가 붙은 식별자입니다. `label` 은 그 노드의 종류(약물·질병·유전자·증상·약효분류)입니다.

In [ ]:
# 파일을 파이썬으로 먼저 열어 본다. 머리글·행 수는 Cypher 를 쓰기 전에 확인하는 것이다
node_lines = open('data/hetionet_nodes.csv', encoding='utf-8').read().splitlines()
print('머리글  :', node_lines[0])
print('첫 행   :', node_lines[1])
print('데이터 행:', len(node_lines) - 1)

In [ ]:
# 관계 파일도 같은 방법으로 본다. 머리글이 노드 파일과 다르다는 것이 요점이다
edge_lines = open('data/hetionet_edges.csv', encoding='utf-8').read().splitlines()
print()
print('관계 머리글  :', edge_lines[0])
print('관계 첫 행   :', edge_lines[1])
print('관계 데이터 행:', len(edge_lines) - 1)

In [ ]:
# 서버 쪽에서도 훑어본다: 아직 노드를 만들지 않고, 파일에 어떤 종류가 몇 개씩 있는지만 센다
for row in run_cypher("LOAD CSV WITH HEADERS FROM 'file:///hetionet_nodes.csv' AS row "
                      "RETURN row.label AS 종류, count(*) AS 개수 ORDER BY 개수 DESC"):
    print(row)

In [ ]:
# 관계 파일도 같은 방법으로 훑는다. 관계 타입이 12종이라는 것을 적재 전에 알 수 있다
for row in run_cypher("LOAD CSV WITH HEADERS FROM 'file:///hetionet_edges.csv' AS row "
                      "RETURN row.rel AS 관계타입, count(*) AS 개수 ORDER BY 개수 DESC"):
    print(row)

> `WITH HEADERS` 는 첫 줄을 **머리글**로 삼아 각 줄을 `{id: ..., name: ..., label: ...}` 꼴의 map 으로 줍니다. 머리글이 영문이라 `row.label` 처럼 점으로 꺼낼 수 있습니다. 한글이거나 공백·괄호가 섞이면 `row['역간거리(km)']` 처럼 **대괄호**로 꺼냅니다.

## 인코딩: 국내 공공데이터의 첫 관문
지하철 파일은 서울 열린데이터광장에서 받은 **바이트 그대로**입니다. 열어 보겠습니다.

In [ ]:
# 국내 공공데이터는 CP949 인코딩이 흔하다. UTF-8 로 읽으면 이 자리에서 멈춘다
try:
    open('data/seoul_metro_stations_cp949.csv', encoding='utf-8').read()
except UnicodeDecodeError as error:
    # 몇 번째 바이트에서 막혔는지가 메시지에 적혀 있다. 인코딩이 다르다는 신호다
    print('UTF-8 로는 못 읽습니다:', error)

In [ ]:
# 인코딩을 cp949 로 지정하면 읽힌다. 읽어서 UTF-8 로 다시 저장해 둔다
metro_text = open('data/seoul_metro_stations_cp949.csv', encoding='cp949').read()
print('머리글:', metro_text.splitlines()[0])
print('첫 행 :', metro_text.splitlines()[1])

In [ ]:
# LOAD CSV 에는 인코딩을 지정하는 자리가 없다. 그래서 적재 전에 파일을 UTF-8 로 바꿔 둔다
open('data/seoul_metro_stations_utf8.csv', 'w', encoding='utf-8').write(metro_text)
print('UTF-8 로 저장했습니다: data/seoul_metro_stations_utf8.csv')

> **`LOAD CSV` 는 UTF-8 만 읽습니다.** 인코딩을 지정하는 옵션이 아예 없습니다. 그래서 포털에서 받은 `CP949` 파일은 **적재 전에 파이썬으로 한 번 열어 UTF-8 로 다시 써 두는** 것이 정석입니다. 이 한 단계를 빠뜨리면 글자가 깨진 채로 적재되거나 파일을 아예 못 읽습니다.

방금 만든 UTF-8 파일을 `import` 폴더로도 보내야 `LOAD CSV` 가 볼 수 있습니다. 아래 복사 셀을 한 번 더 실행하세요(맨 위에서 실행했던 그 셀입니다).

In [ ]:
# [제공 코드] 데이터 파일을 Neo4j import 폴더로 복사: 이 셀은 실행만 하세요.
# LOAD CSV·apoc.load.json 은 보안상 서버의 import 폴더 안 파일만 읽습니다.
# - .env 에 NEO4J_IMPORT_DIR 이 있으면 data/ 의 파일을 자동 복사합니다.
# - 없으면(수동 복사한 경우) 그대로 넘어갑니다. Neo4j Desktop 은 인스턴스 메뉴의
#   "Open folder > Import" 로 폴더를 열어 data/ 의 파일을 직접 복사해 두세요.
import shutil
from pathlib import Path

_IMPORT_DIR = os.getenv("NEO4J_IMPORT_DIR", "")
_SRC_DIR = Path("data") if Path("data").exists() else Path("../data")
if _IMPORT_DIR:
    # csv 와 json 만 복사합니다. 다음 준비 셀이 이 파일들을 file:/// 로 읽습니다
    for f in sorted(_SRC_DIR.glob("*.csv")) + sorted(_SRC_DIR.glob("*.json")):
        shutil.copy(f, Path(_IMPORT_DIR) / f.name)
        print("복사:", f.name)
else:
    print("NEO4J_IMPORT_DIR 미설정: data/ 의 파일을 import 폴더에 직접 복사했는지 확인하세요.")

### 🖐️ 함께 따라하기: 지하철 파일 훑어보기

데모는 의료 데이터였죠. 따라하기는 방금 UTF-8 로 바꾼 **`seoul_metro_stations_utf8.csv`** 로 연습합니다. `LOAD CSV WITH HEADERS` 로 읽어 **호선별 역 수**를 많은 순으로 세어 출력하세요. 머리글이 한글이므로 **대괄호**로 꺼내야 합니다(`row['호선']`). 변수 이름은 `line_counts` 로 합니다.

**확인 기준**: 호선이 8개 나오고, 가장 많은 호선은 56개입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) LOAD CSV WITH HEADERS FROM 'file:///seoul_metro_stations_utf8.csv' AS row 로 읽는다
# 2) RETURN row['호선'] AS 호선, count(*) AS 역수 ORDER BY 역수 DESC 로 세어 line_counts 에 담는다
# 3) for 문으로 한 줄씩 출력한다

### ✅ 바로 확인 퀴즈

**1.** `LOAD CSV WITH HEADERS` 로 읽으면 `역간거리(km)` 칸의 `1.1` 은 어떤 자료형으로 들어오나요?

<details><summary>정답 보기</summary>

**문자열(`"1.1"`)** 입니다. CSV 의 모든 값은 문자열이라, 숫자로 쓰려면 `toFloat()`(정수는 `toInteger()`)로 변환해야 합니다.

</details>

**2.** 포털에서 받은 `CP949` 파일을 `LOAD CSV` 로 바로 읽을 수 있나요?

<details><summary>정답 보기</summary>

없습니다. `LOAD CSV` 에는 **인코딩을 지정하는 옵션이 없고 UTF-8 만** 읽습니다. 파이썬으로 `encoding='cp949'` 로 열어 UTF-8 로 다시 저장한 뒤 그 파일을 적재합니다.

</details>

---
# 2. 제약조건을 먼저 걸고 노드를 적재합니다

## 왜 필요할까요?
모델을 정했으니 이제 적재합니다. 순서는 지난 교안에서 익힌 그대로입니다. **제약조건을 먼저 겁니다.**

- 나중에 걸면 이미 들어간 중복 때문에 **제약 생성 자체가 실패**합니다(교안_01 1절에서 그 이유를 짚었습니다).
- 제약이 만드는 인덱스가 없으면 `MERGE` 가 매번 전체를 훑어 **적재가 크게 느려집니다**(3절에서 잽니다).

## 인덱스는 레이블마다 따로 걸립니다
여기가 오늘의 핵심입니다. `CREATE CONSTRAINT ... FOR (n:Compound) REQUIRE n.id IS NODE KEY` 는 **`:Compound` 레이블이 붙은 노드의 `id`** 에만 걸립니다. `:Disease` 는 별개입니다. 그래서 레이블이 5종이면 **제약도 5개**를 겁니다.

> **키 제약(`IS NODE KEY`)은 Enterprise 기능입니다.** 이 수업은 Neo4j Desktop 으로 실습하고, Desktop 에는 Enterprise 개발자 라이선스가 딸려 옵니다. Docker 나 Homebrew 의 기본 이미지로 돌리면 `Node Key constraint requires Neo4j Enterprise Edition` 에서 멈춥니다. 그때는 `IS NODE KEY` 를 `IS UNIQUE` 로 바꾸면 적재는 되지만, **키가 비어 있는 노드를 막아 주지는 못합니다**(앞 교안에서 본 그 구멍입니다).

In [ ]:
# 레이블 5종에 각각 제약을 건다. 레이블은 값이 아니라 쿼리의 구조라 문자열로 이어 붙인다
labels = ['Compound', 'Disease', 'Gene', 'Symptom', 'PharmacologicClass']
for label in labels:
    # 제약 이름은 레이블마다 다르게 소문자로 붙인다
    run_cypher(f"CREATE CONSTRAINT {label.lower()}_id IF NOT EXISTS "
               f"FOR (n:{label}) REQUIRE n.id IS NODE KEY")

for row in run_cypher("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties "
                      "RETURN name, labelsOrTypes, properties ORDER BY name"):
    print(row)

> `IF NOT EXISTS` 를 붙이면 이미 있어도 에러 없이 넘어가, 노트북을 다시 실행해도 안전합니다.

## 적재 전에 오염 행을 세어 둡니다
적재 쿼리에서 걸러 내기 전에, **걸러질 행이 몇 개인지 먼저 셉니다.** 0이면 데이터가 깨끗한 것이고, 0이 아니면 왜 그런지 알아야 합니다. "몇 개가 버려지는지 모르는 채로 적재하는 것"이 가장 위험합니다.

In [ ]:
# 식별자(id)가 비어 있는 행이 몇 개인지 적재 전에 센다
# count(CASE WHEN ... THEN 1 END): 조건에 맞는 행만 센다(아니면 null 이라 세지 않는다)
check = run_cypher("""
LOAD CSV WITH HEADERS FROM 'file:///hetionet_nodes.csv' AS row
RETURN count(*) AS 전체행,
       count(CASE WHEN trim(coalesce(row.id, '')) = '' THEN 1 END) AS 빈식별자,
       count(CASE WHEN trim(coalesce(row.name, '')) = '' THEN 1 END) AS 빈이름
""")[0]
print(check)

## 문법: LOAD CSV + 값으로 레이블 정하기 + 배치 커밋
```cypher
LOAD CSV WITH HEADERS FROM 'file:///hetionet_nodes.csv' AS row
WITH row WHERE trim(coalesce(row.id, '')) <> ''
CALL (row) {
  MERGE (n:$(row.label) {id: row.id})
  SET n.name = row.name
} IN TRANSACTIONS OF 5000 ROWS
```
- **`WITH row WHERE ...`**: `LOAD CSV` 바로 뒤에는 `WHERE` 를 붙일 수 없습니다. `WITH row` 로 한 번 받아 넘겨야 거를 수 있습니다. `trim(coalesce(x, ''))` 는 빈 칸(`null`)과 공백만 든 칸을 함께 막습니다.
- **`MERGE (n:$(row.label) {id: row.id})`**: 한 쿼리로 **행마다 다른 레이블**을 붙입니다. `label` 칸이 `Compound` 인 행은 `(:Compound)`, `Disease` 인 행은 `(:Disease)` 가 됩니다. 속성 값(`row.id`)은 원래부터 행에서 읽어 오지만, **레이블은 이름 자리라 글자로 박아야 했습니다.** `$(...)` 가 "이 이름도 행에서 읽어라"라는 표시입니다(Neo4j 5.26 이상). 이 표시가 없으면 `:Compound` 만 넣는 쿼리, `:Disease` 만 넣는 쿼리 하는 식으로 **레이블 종류 수만큼(여기서는 5개)** 따로 써야 합니다. `$(...)` 없이 `(n:row.label)` 이라고 쓰면 **문법 오류**입니다(레이블 자리는 이름을 적는 자리라 점을 못 받습니다).
- **`CALL (row) { ... } IN TRANSACTIONS OF 5000 ROWS`**: 5,000행마다 **커밋**(그때까지의 변경을 확정)하고 다음 묶음으로 넘어갑니다. 괄호 안의 `(row)` 는 "이 변수만 안으로 들여보낸다"는 뜻입니다.

<img src="images/batch_commit_timeline.png" width="760">

In [ ]:
# 노드 15,540행을 5,000행씩 끊어 적재한다($(row.label) 로 레이블을 정하고, WHERE 로 빈 id 를 거른다)
import time

started = time.time()
run_cypher("""
LOAD CSV WITH HEADERS FROM 'file:///hetionet_nodes.csv' AS row
WITH row WHERE trim(coalesce(row.id, '')) <> ''
CALL (row) {
  MERGE (n:$(row.label) {id: row.id})
  SET n.name = row.name
} IN TRANSACTIONS OF 5000 ROWS
""")
node_seconds = time.time() - started
print(f"적재 시간: {node_seconds:.2f}초")

In [ ]:
# 적재했으면 숫자로 검증한다. 파일 행 수와 노드 수가 같아야 한다
n_node = run_cypher("MATCH (n) RETURN count(n) AS n")[0]['n']
print('파일 행 수:', len(node_lines) - 1)
print('노드 수   :', n_node)
print('일치     :', n_node == len(node_lines) - 1)

In [ ]:
# 종류별로도 대조한다. 1절에서 파일을 훑어 본 수와 같아야 한다
for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS 종류, count(*) AS 개수 "
                      "ORDER BY 개수 DESC"):
    print(row)

In [ ]:
# 같은 적재를 한 번 더: MERGE 라서 건수가 그대로여야 한다(멱등)
# CREATE 였다면 같은 노드가 한 벌 더 생겨 노드 수가 두 배가 된다
run_cypher("""
LOAD CSV WITH HEADERS FROM 'file:///hetionet_nodes.csv' AS row
WITH row WHERE trim(coalesce(row.id, '')) <> ''
CALL (row) {
  MERGE (n:$(row.label) {id: row.id})
  SET n.name = row.name
} IN TRANSACTIONS OF 5000 ROWS
""")
print('재적재 후 노드 수:', run_cypher("MATCH (n) RETURN count(n) AS n")[0]['n'], '(그대로면 멱등 성공)')

### 🖐️ 함께 따라하기: 지하철 역 노드 적재하기

이제 지하철 역을 적재합니다. **역명으로 `MERGE`** 합니다. 환승역은 호선마다 줄이 따로 있지만 역은 하나이므로, 이름이 같으면 같은 노드여야 합니다.

1. 레이블 `Station` 의 `name` 에 **키 제약**을 이름 `station_name` 으로 만드세요(`IF NOT EXISTS` 포함).
2. `seoul_metro_stations_utf8.csv` 를 `LOAD CSV` 로 읽어 `:Station` 노드를 적재하세요. `CALL (row) { ... } IN TRANSACTIONS OF 100 ROWS` 를 씁니다.
3. 적재된 역 수를 `n_station` 에 담아 `역 노드: N개 (파일 279행)` 꼴로 출력하세요.

**확인 기준**: 파일은 279행인데 역 노드는 **241개**입니다. 줄어든 38개는 **같은 역이 여러 줄에 나온 몫**입니다(이름이 겹치는 역이 35곳인데, 두 번 나오는 역 32곳에서 32줄, 세 번 나오는 역 3곳에서 6줄이 겹쳐 38줄이 줄어듭니다). 그 35곳 중 34곳은 환승역이고, 한 곳(응암)은 6호선 순환 구간이라 한 호선에 두 번 적힌 것입니다. **건수가 줄었다고 늘 사고인 것은 아닙니다. 왜 줄었는지 설명할 수 있으면 됩니다.**

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE CONSTRAINT station_name IF NOT EXISTS FOR (s:Station) REQUIRE s.name IS NODE KEY
# 2) LOAD CSV ... CALL (row) { MERGE (s:Station {name: row['역명']}) } IN TRANSACTIONS OF 100 ROWS
# 3) MATCH (s:Station) RETURN count(s) 로 n_station 을 구해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 레이블이 5종인데 제약을 `:Compound` 하나에만 걸면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

인덱스는 **레이블마다 따로** 걸립니다. `:Disease`·`:Gene` 등 나머지 4종에는 인덱스가 없어, 그 노드들을 `MERGE` 하거나 찾을 때 **전부 훑게** 됩니다. 유일성도 그 레이블에는 보장되지 않습니다.

</details>

**2.** `MERGE (n:$(row.label) {id: row.id})` 에서 `$(...)` 는 무엇을 하나요?

<details><summary>정답 보기</summary>

레이블 이름을 **데이터의 값에서** 가져옵니다. `label` 칸이 `Disease` 인 행은 `:Disease` 가 붙습니다. 이게 없으면 레이블 종류마다 쿼리를 따로 써야 합니다(Neo4j 5.26 이상).

</details>

**3.** `IN TRANSACTIONS OF 5000 ROWS` 를 붙이면 무엇이 달라지나요?

<details><summary>정답 보기</summary>

거대한 변경을 **한 트랜잭션에 쥐고 있다가 메모리가 터지는 문제**를 막습니다. 5,000행마다 커밋하고 다음 배치로 넘어갑니다. 중간에 실패해도 그때까지 커밋된 배치는 남습니다(그래서 `MERGE` 로 멱등하게 짜 두면 실패 지점부터 다시 돌릴 수 있습니다).

</details>

---
# 3. 관계를 적재합니다

## 왜 필요할까요?
이제 관계 91,966개를 적재합니다. 관계는 노드와 달리 **양 끝을 먼저 찾아야** 만들 수 있습니다(교안_01 3-1).
```cypher
MATCH (a:Compound {id: row.source})
MATCH (b:Disease {id: row.target})
MERGE (a)-[:TREATS]->(b)
```
양 끝에 **레이블을 적습니다.** 2절에서 건 제약의 인덱스는 **`(:Compound).id`** 처럼 레이블에 걸려 있어서, 레이블을 적어야 그 인덱스를 탑니다. 레이블 없이 `MATCH (a {id: row.source})` 라고 쓰면 한 행마다 **그래프의 모든 노드를 훑습니다**(지금 1만 5천 개가 넘습니다). 31일차에서 인덱스로 확인한 그 차이가, 적재에서는 **행마다** 되풀이됩니다.

양 끝의 레이블은 `data/hetionet_relspec.json` 에 타입마다 적혀 있습니다(`rel`·`source_label`·`target_label`). 이것을 읽어 **타입마다 한 번씩** 적재하겠습니다.

In [ ]:
# 관계 타입 -> (출발 레이블, 도착 레이블) 대응표를 읽는다
# 이 명세서 하나로 적재(3절)와 검증(4절)을 둘 다 돌린다. 실무에서 쓰는 방식이다
import json

with open('data/hetionet_relspec.json', encoding='utf-8') as f:
    rel_spec = json.load(f)

for spec in rel_spec:
    print(spec)

In [ ]:
# 이 수만큼 아래 적재를 돌린다. 명세서 한 줄이 곧 적재 한 번이다
print('관계 타입 수:', len(rel_spec))

In [ ]:
# 타입마다 한 번씩, 양 끝 레이블을 지정해 적재한다
# 레이블·관계 타입은 값이 아니라 쿼리의 '구조'라 $파라미터에 못 넣는다(문자열로 만들어 넣는다)
started = time.time()
for spec in rel_spec:
    run_cypher(
        "LOAD CSV WITH HEADERS FROM 'file:///hetionet_edges.csv' AS row "
        "WITH row WHERE row.rel = $rel "          # 값이므로 이쪽은 파라미터로 넘긴다
        "CALL (row) { "
        f"  MATCH (a:{spec['source_label']} {{id: row.source}}) "
        f"  MATCH (b:{spec['target_label']} {{id: row.target}}) "
        f"  MERGE (a)-[:{spec['rel']}]->(b) "
        # 관계도 5,000건마다 커밋한다. 노드 적재와 같은 기준이다
        "} IN TRANSACTIONS OF 5000 ROWS",
        rel=spec['rel'])
edge_seconds = time.time() - started
print(f"관계 {len(rel_spec)}종 적재: {edge_seconds:.2f}초")

In [ ]:
# 관계도 건수로 검증한다. 파일 행 수와 같아야 한다
n_edge = run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]['n']
print('파일 행 수:', len(edge_lines) - 1)
print('관계 수   :', n_edge)
print('일치     :', n_edge == len(edge_lines) - 1)

> **양 끝 노드를 못 찾은 행은 조용히 사라집니다.** `MATCH` 가 실패하면 그 행은 에러 없이 버려집니다. 레이블을 잘못 적었다면 관계가 0개 만들어지고도 쿼리는 "성공"으로 끝납니다. 그래서 관계 적재는 **반드시 건수로 대조**해야 합니다.

**배치 크기는 어떻게 고르나요?** 정답은 없고 1,000~10,000행이 흔한 범위입니다. 작게 잡으면 커밋 횟수가 늘어 느려지고, 크게 잡으면 한 트랜잭션이 쥐는 메모리가 늘어 위험해집니다. **먼저 인덱스부터 걸고**, 그다음 데이터·장비에 맞춰 조정하는 순서로 접근합니다.

### 🖐️ 함께 따라하기: 호선 노드와 운행 관계 적재하기

지하철 그래프에 호선을 붙입니다. 호선은 여러 역이 나눠 갖는 값이라 속성이 아니라 노드로 둡니다. `(:Line)-[:SERVES]->(:Station)` 입니다.

1. 레이블 `Line` 의 `name` 에 **키 제약**을 이름 `line_name` 으로 만드세요.
2. `seoul_metro_stations_utf8.csv` 를 읽어 `:Line` 노드를 `MERGE` 하고, **역은 이미 만들어 두었으므로 `MATCH` 로 찾아** `SERVES` 관계를 만드세요. 관계 속성으로 `seq`(연번, `toInteger`)를 넣습니다. `IN TRANSACTIONS OF 100 ROWS` 를 씁니다.
3. 호선 수와 관계 수를 각각 `n_line`·`n_serves_all` 에 담아 `호선 N개 · 운행 관계 N개` 꼴로 출력하세요.

**확인 기준**: 호선 **8개**, 운행 관계 **277개** 입니다. 파일은 279행인데 관계가 2건 적습니다. **2호선 본선은 순환선**이라 `시청` 이 시작과 끝에 두 번 나오고, **6호선은 서쪽 끝에 응암순환 구간**이 있어 `응암` 이 두 번 나오기 때문입니다. `MERGE (l)-[:SERVES]->(s)` 는 같은 호선·역 쌍에 관계를 하나만 만듭니다.

> 2절에서 역 수가 줄었을 때와 같은 이야기입니다. **줄었다고 늘 사고인 것은 아니고, 왜 줄었는지 설명할 수 있으면 됩니다.** 반대로 설명이 안 되는 감소는 `MATCH` 가 못 찾은 행이 있다는 신호입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE CONSTRAINT line_name IF NOT EXISTS FOR (l:Line) REQUIRE l.name IS NODE KEY
# 2) LOAD CSV ... CALL (row) { MERGE (l:Line {name: row['호선']})
#                              MATCH (s:Station {name: row['역명']})
#                              MERGE (l)-[r:SERVES]->(s) SET r.seq = toInteger(row['연번']) }
#    IN TRANSACTIONS OF 100 ROWS
# 3) count 두 번으로 n_line·n_serves_all 을 구해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 관계 적재에서 `MATCH (a {id: row.source})` 처럼 **레이블을 빼면** 왜 느린가요?

<details><summary>정답 보기</summary>

인덱스는 **레이블마다** 걸립니다. 레이블을 쓰지 않으면 어느 인덱스도 쓸 수 없어, 한 행마다 그래프의 **모든 노드를 훑어** `id` 를 비교합니다. 노드가 많아질수록 급격히 느려집니다.

</details>

**2.** 관계 적재 쿼리에서 레이블을 **틀리게** 적으면(예: `:Disease` 자리에 `:Gene`) 어떻게 되나요?

<details><summary>정답 보기</summary>

`MATCH` 가 아무것도 못 찾아 그 행이 **조용히 버려집니다.** 에러 없이 쿼리는 성공으로 끝나고 관계만 안 만들어집니다. 그래서 적재 후 **건수를 파일 행 수와 대조**해야 합니다.

</details>

**3.** 배치 크기를 1로 잡으면 어떤 점이 나빠질까요?

<details><summary>정답 보기</summary>

행마다 커밋이 일어나 **커밋 횟수가 행 수만큼** 늘어납니다. 메모리는 아주 적게 쓰지만 그만큼 느려집니다. 메모리와 속도 사이의 손잡이라고 보면 됩니다.

</details>

## 배치는 파일 적재 전용이 아닙니다
`CALL (변수) { ... } IN TRANSACTIONS` 에 들여보내는 것이 꼭 파일의 행일 필요는 없습니다. **앞에서 `MATCH` 한 노드**를 그대로 들여보낼 수 있습니다. 이미 적재해 둔 수만 개 노드에 값을 채워 넣거나 고칠 때 쓰는 형태이고, 한 트랜잭션에 다 담으면 메모리가 터지는 것은 적재 때와 똑같습니다.

In [ ]:
# 들여보내는 것이 파일 행이 아니라 MATCH 결과일 뿐, 문법은 2절과 같다
# 이미 그래프에 있는 약물 1,531개에 '이 데이터가 어디서 왔는지' 를 한꺼번에 적어 둔다
run_cypher("""
MATCH (n:Compound)
CALL (n) {
  SET n.dataset = 'hetionet'
} IN TRANSACTIONS OF 5000 ROWS
""")
tagged = run_cypher("MATCH (n:Compound) WHERE n.dataset = 'hetionet' RETURN count(n) AS n")[0]['n']
print('출처를 적어 둔 약물:', tagged)

---
# 4. APOC 로 명세서와 대조하고 스키마를 훑습니다

## 왜 필요할까요?
적재가 끝나면 두 가지를 하고 싶습니다. "그래프에 지금 무엇이 얼마나 있나" 한눈에 보기, 그리고 **"들어가야 할 만큼 들어갔나" 명세서와 대조하기**입니다. 이럴 때 쓰는 확장 도구 모음이 **APOC** 입니다.

## 문법
- **`apoc.meta.stats()`**: 레이블별 노드 수·관계 타입별 개수 등 **그래프 스키마 통계**를 줍니다.
- **`apoc.load.json('file:///파일.json')`**: JSON 파일을 읽어 `value` 로 한 건씩 돌려줍니다. CSV 와 달리 **중첩된 구조**를 그대로 담을 수 있어, 공공데이터가 JSON 으로 오는 경우가 많습니다.

In [ ]:
# apoc.meta.stats() 로 그래프 스키마를 한눈에 훑는다
# 노드를 하나씩 세는 것이 아니라 서버가 들고 있는 통계를 읽는 것이라 큰 그래프에서도 빠르다
stats = run_cypher("CALL apoc.meta.stats() YIELD labels, relTypesCount, nodeCount, relCount "
                   "RETURN labels, relTypesCount, nodeCount, relCount")[0]
print('전체 노드 수:', stats['nodeCount'])
print('전체 관계 수:', stats['relCount'])
print('레이블별    :', stats['labels'])

In [ ]:
# 명세서 JSON 을 읽어 '들어가야 할 건수'와 '실제 건수'를 대조한다
# apoc.load.json 은 배열의 원소를 value 로 하나씩 돌려준다
for row in run_cypher("""
CALL apoc.load.json('file:///hetionet_relspec.json') YIELD value
CALL (value) {
  MATCH ()-[r]->() WHERE type(r) = value['rel']
  RETURN count(r) AS 실제
}
RETURN value['rel'] AS 관계타입, value['expected'] AS 명세, 실제, (실제 = value['expected']) AS 일치
ORDER BY 관계타입
"""):
    print(row)

> 12줄 모두 `일치: True` 여야 합니다. 한 줄이라도 어긋나면 그 타입의 적재가 조용히 실패한 것입니다. **적재 스크립트와 검증 스크립트를 같은 명세서로 돌리는 것**이 실무에서 쓰는 방식입니다. 3절의 적재 루프도 이 파일을 읽었습니다.

> 위 쿼리의 `CALL (value) { ... }` 는 2절의 `CALL (row) { ... }` 와 같은 문법입니다. `IN TRANSACTIONS` 없이 쓰면 "각 행마다 이 부분을 따로 계산해서 붙여라"라는 뜻이 됩니다.

### 🖐️ 함께 따라하기: 노선 요약 JSON 붙이기

`seoul_metro_lines.json` 에는 노선별 요약이 **중첩 구조**로 들어 있습니다.

```json
{"호선": "1", "역수": 10, "총연장": 7.8, "기점": "서울역", "종점": "청량리",
 "역목록": ["서울역", "시청", ...]}
```

`apoc.load.json` 으로 읽어, 앞에서 만든 `:Line` 노드에 `length_km`(총연장, `toFloat`)와 `station_count`(역수, `toInteger`)를 붙이세요. 그다음 **총연장이 가장 긴 노선**을 `가장 긴 노선: N호선 (N km)` 꼴로 출력합니다(변수 이름 `longest_line`).

**확인 기준**: 총연장이 가장 긴 노선은 **2호선**이고 **60.2km** 입니다.

> `:Line` 노드는 이미 있으므로 `MERGE` 로 찾아 `SET` 합니다(`CREATE` 면 노선이 중복 생성됩니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CALL apoc.load.json('file:///seoul_metro_lines.json') YIELD value 로 읽는다
# 2) MERGE (l:Line {name: value['호선']}) 하고 length_km·station_count 를 SET 한다
# 3) MATCH (l:Line) ... ORDER BY 총연장 DESC LIMIT 1 로 longest_line 을 구해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 적재가 끝난 뒤 `apoc.meta.stats()` 로 무엇을 확인할 수 있나요?

<details><summary>정답 보기</summary>

레이블별 노드 수·관계 타입별 개수·전체 노드/관계 수 등 **그래프의 스키마 통계**입니다. 파일에서 미리 센 수와 대조하면 적재가 제대로 끝났는지 알 수 있습니다.

</details>

**2.** 같은 데이터를 CSV 대신 JSON 으로 받는 게 나을 때는 언제인가요?

<details><summary>정답 보기</summary>

**중첩 구조**가 있을 때입니다. 노선 하나에 역 목록이 딸린 것처럼 "한 건 안에 여러 건"이 들어 있으면 CSV 로는 행을 쪼개 표현해야 하지만 JSON 은 그대로 담깁니다.

</details>

---
# 5. 다른 길: 파이썬이 읽어 보냅니다

## 왜 필요할까요?
여기까지는 **서버가 파일을 읽었습니다.** `LOAD CSV` 도 `apoc.load.json` 도 서버의 `import` 폴더를 들여다봅니다. 그런데 서버가 그 폴더를 못 보는 경우가 있습니다. 관리형 서비스라 접근 권한이 없거나, 원본이 애초에 파일이 아니라 API 응답일 때입니다.

그럴 때 쓰는 두 번째 길이 **파이썬이 읽어 보내는 것**입니다. 사실 교안_01 2-2 에서 이미 해 봤습니다. 파이썬 리스트를 `$rows` 로 통째로 넘기고 `UNWIND` 가 그것을 행으로 펴는 방법이었죠. 그때와 달라지는 것은 그 리스트를 **손으로 적느냐 파일에서 읽느냐** 한 줄뿐입니다.

그림으로 정리하면 이렇습니다.

<img src="images/two_load_paths.png" width="880">

왼쪽이든 오른쪽이든, 화살표가 같은 노드에서 만납니다.

## 문법: 파일 -> 리스트 -> UNWIND
```python
rows = pd.read_csv('data/파일.csv', dtype=str).to_dict('records')
run_cypher("UNWIND $rows AS row "
           "MERGE (n:$(row.label) {id: row.id}) SET n.name = row.name", rows=rows)
```
2절에서 서버가 읽었던 그 15,540행을, 이번에는 파이썬이 읽어 보냅니다. **같은 데이터를 다른 길로 다시 넣는 것**이라, 끝나고 그래프가 그대로면 두 길이 같은 곳에 도착한다는 뜻입니다.

In [ ]:
# 파이썬이 파일을 읽는다. 이 길에서는 인코딩·전처리를 파이썬 쪽에서 해결한다
import time

import pandas as pd

# to_dict('records'): 한 행을 사전으로 만든다. 그 목록이 그대로 쿼리 파라미터가 된다
# (dtype=str 로 읽어 CSV 의 값이 전부 문자열이라는 성질을 그대로 가져간다)
node_rows = pd.read_csv('data/hetionet_nodes.csv', dtype=str).to_dict('records')
print('읽은 행:', len(node_rows))
print('첫 행  :', node_rows[0])

In [ ]:
# 리스트를 통째로 넘겨 UNWIND 로 일괄 적재(한 번의 왕복)
# 2절의 LOAD CSV 와 달리 파일은 파이썬이 읽고, 서버는 받은 목록만 UNWIND 로 처리한다
started = time.time()
run_cypher("UNWIND $rows AS row "
           "MERGE (n:$(row.label) {id: row.id}) "
           "SET n.name = row.name", rows=node_rows)
print(f"적재 시간: {time.time() - started:.2f}초")

In [ ]:
# id 가 있는 노드만 센다. 따라하기에서 만든 역·호선 노드(id 가 없다)는 빠진다
loaded = run_cypher("MATCH (n) WHERE n.id IS NOT NULL RETURN count(n) AS n")[0]['n']
print('id 를 가진 노드:', loaded)

> **15,540개 그대로입니다.** 위 그림처럼 **같은 노드**에 도착했다는 뜻입니다. `MERGE` 가 새로 만들지 않았습니다. 두 길은 도구가 다를 뿐 도착지가 같습니다.

> 늘었다면 둘 중 하나가 틀린 것입니다. 키를 다르게 잡았거나(`id` 대신 다른 칸), 레이블을 다르게 붙였거나. **같은 데이터를 두 경로로 넣어 보고 건수가 그대로인지 보는 것**은 실무에서 적재 코드를 바꿀 때 쓰는 점검법이기도 합니다.

> **목록이 길면 끊어 넣습니다.** 위 셀은 15,540건을 **한 트랜잭션**으로 넣었습니다. 건수가 수십만이 되면 그 하나가 쥐는 메모리가 문제가 됩니다. 2절의 `LOAD CSV` 에서 쓴 `IN TRANSACTIONS OF 5000 ROWS` 를 **이 길에서도 그대로** 쓸 수 있습니다. 파일에서 온 행이든 파라미터로 받은 목록이든 `CALL (row) { ... }` 안쪽은 같습니다.

In [ ]:
# 같은 적재를 5,000건씩 끊어 커밋한다. 목록이 길어질수록 이 방식이 안정적이다
# MERGE 라서 이미 들어간 노드는 다시 만들어지지 않는다
started = time.time()
run_cypher("UNWIND $rows AS row "
           "CALL (row) { "
           "  MERGE (n:$(row.label) {id: row.id}) "
           "  SET n.name = row.name "
           "} IN TRANSACTIONS OF 5000 ROWS", rows=node_rows)
print(f"배치로 다시 적재: {time.time() - started:.2f}초")

In [ ]:
# 이 길의 또 다른 이점: 결과가 파이썬 자료구조로 돌아온다
# run_cypher 는 dict 의 리스트를 주므로 그대로 DataFrame 이 된다
df = pd.DataFrame(run_cypher("MATCH (n) WHERE n.id IS NOT NULL "
                            "RETURN labels(n)[0] AS 종류, count(*) AS 개수 "
                            "ORDER BY 개수 DESC"))
display(df)

> 이제 정렬·필터·통계 같은 pandas 스킬을 그대로 쓸 수 있습니다. 그래프에서 뽑은 결과를 **분석 도구로 넘기는 다리**가 이 한 줄입니다.

### 🖐️ 함께 따라하기: CP949 파일을 파이썬이 읽어 적재하기

1절에서는 `CP949` 파일을 UTF-8 로 **바꿔 두어야** `LOAD CSV` 가 읽을 수 있었습니다. 이 길에서는 파이썬이 읽으므로 **인코딩만 지정하면 끝**입니다. 변환 파일을 만들 필요가 없습니다.

1. `seoul_metro_stations_cp949.csv` 를 **`encoding='cp949'`** 로 읽어 `metro_rows`(dict 리스트)에 담으세요.
2. `$rows` 로 넘겨 `UNWIND` 로 `:Station` 노드를 적재하세요(`MERGE (s:Station {name: row['역명']})`).
3. 적재 결과를 세어 `파일 279행 -> 역 노드 N개` 꼴로 출력하세요(변수 이름 `station_count`).

**확인 기준**: 파일 279행에서 역 노드 **241개**입니다. 2절에서 `LOAD CSV` 로 넣은 수와 **같아야** 합니다. `MERGE` 라서 늘지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) seoul_metro_stations_cp949.csv 를 encoding='cp949' 로 읽어 metro_rows(dict 리스트)에 담는다
# 2) run_cypher 로 UNWIND $rows AS row MERGE (s:Station {name: row['역명']}) 를 실행한다
# 3) MATCH (s:Station) RETURN count(s) 로 station_count 를 구해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 같은 `UNWIND $rows` 인데, 2절의 `LOAD CSV` 와 이 절의 방법은 무엇이 다른가요?

<details><summary>정답 보기</summary>

**파일을 읽는 주체**가 다릅니다. `LOAD CSV` 는 서버가 `import` 폴더의 파일을 읽지만, 이 절은 **파이썬이 파일을 읽어** 리스트로 만든 뒤 파라미터로 보냅니다. 그래서 인코딩·전처리를 파이썬에서 먼저 할 수 있습니다. 묶어 보내 왕복을 줄이는 것은 두 경로가 같습니다.

</details>

**2.** `CP949` 파일을 다룰 때 두 경로는 무엇이 다른가요?

<details><summary>정답 보기</summary>

`LOAD CSV` 는 인코딩 옵션이 없어 **UTF-8 로 바꾼 파일을 따로 만들어야** 합니다(1절에서 그렇게 했습니다). 파이썬 경로는 `encoding='cp949'` 한 줄로 끝납니다.

</details>

**3.** 다시 적재했는데 노드 수가 **늘었다면** 무엇을 의심해야 하나요?

<details><summary>정답 보기</summary>

두 경로가 **같은 키로 `MERGE` 하고 있지 않다**는 뜻입니다. 키가 다르거나 레이블이 다르면 같은 데이터가 다른 노드로 한 벌 더 생깁니다. 건수가 그대로여야 두 길이 같은 곳에 도착한 것입니다.

</details>

---
## 어느 길로 적재할까요? LOAD CSV 와 파이썬

이 단원은 파일을 그래프로 넣는 길을 **두 가지** 배웠습니다. 교재 밖에서 멈추지 않으려면 언제 어느 쪽을 쓰는지 정해 두어야 합니다.

| 상황 | `LOAD CSV` (서버가 읽음) | 파이썬 + `UNWIND` (파이썬이 읽어 보냄) |
|---|---|---|
| 파일을 서버 `import` 폴더에 놓을 수 있다 | 적합 | 가능 |
| import 폴더에 접근할 수 없는 관리형 서비스 | 불가 | **적합** |
| 원본이 파일이 아니라 API 응답·DB 조회 결과 | 불가 | **적합** |
| 파일 인코딩이 `CP949` 다 | 변환 파일을 따로 만들어야 함 | **`encoding=` 한 줄** |
| 앞뒤 행을 견줘야 만들 수 있는 관계 | 어려움 | **적합**(파이썬에서 쌍을 먼저 만든다) |
| 수십만 행을 한 번에 밀어 넣기 | **적합**(`IN TRANSACTIONS`) | 가능(리스트가 메모리에 다 올라온다. `IN TRANSACTIONS` 도 그대로 쓸 수 있다) |
| 오염 행 걸러 내기 | Cypher 의 `WITH row WHERE` | 파이썬의 리스트 컴프리헨션 |

정리하면 **파일을 서버에 둘 수 있고 양이 아주 많으면 `LOAD CSV`**, **파이썬이 먼저 손을 대야 하거나 원본이 파일이 아니면 파이썬**입니다. 어느 쪽이든 **적재 뒤에 건수로 검증**하는 것은 같습니다.

---
## 이번 강의 정리

| 단계 | 하는 일 | 핵심 |
|---|---|---|
| 파일 훑기 | 머리글·인코딩·행 수 | `CP949` 는 파이썬으로 UTF-8 로 바꿔 둔다 |
| 제약조건 | 적재 **전에**, **레이블마다** | `CREATE CONSTRAINT ... IS NODE KEY` |
| 노드 적재 | 값으로 레이블을 정해 한 번에 | `MERGE (n:$(row.label) {id: row.id})` |
| 배치 커밋 | 수만 행을 끊어 커밋 | `CALL (row) { ... } IN TRANSACTIONS OF n ROWS` |
| 관계 적재 | 타입마다 양 끝 레이블을 지정 | 레이블 한 단어가 **398배** |
| 검증 | 파일 행 수와 그래프 건수를 대조 | 못 찾은 행은 **조용히 사라진다** |
| APOC | 스키마 훑기·JSON 적재·명세서 대조 | `apoc.meta.stats`·`apoc.load.json` |
| 두 번째 길 | 파이썬이 읽어 `$rows` 로 보낸다 | 인코딩·전처리를 파이썬이 먼저 한다 |

- 파일 값은 **모두 문자열**: 숫자는 `toInteger`/`toFloat` 로 바꿉니다(빈 칸은 `null` 이 됩니다).
- `LOAD CSV`·`apoc.load.json` 은 서버 **`import` 폴더**의 파일만, 그리고 **UTF-8 만** 읽습니다.
- 적재는 항상 **건수로 검증**하고, `MERGE` 로 **여러 번 실행해도 안전**하게 만듭니다.
- **앞뒤 행을 견줘야 하는 관계**는 파이썬에서 쌍을 만들어 `UNWIND` 로 보냅니다.

## ⏭️ 예고: 다음 시간

오늘 노드 15,540개와 관계 91,966개짜리 의료 지식그래프를 **만들고 채웠습니다.** 지하철 역과 호선도 얹었습니다. 다음 시간부터는 이 그래프를 **분석**합니다. 어떤 노드가 네트워크에서 중요한지, 서로 얼마나 촘촘히 뭉쳐 있는지 같은 물음을 다룹니다. 관계 타입을 `_CG`·`_DG` 로 나눠 두고 약효분류를 노드로 둔 이 모델이 거기서 다시 한번 값을 합니다.

> 다음 교안은 준비 셀에서 그래프를 다시 비우고 같은 데이터를 새로 적재합니다. 오늘 만든 그래프를 그대로 이어받지 않으니, 이 노트북을 여기서 닫아도 됩니다.

수고하셨습니다!